# Embeddings, Cosine Similarity & Simple RAG Playground

This notebook is a learning playground for the things I have been experimenting with.

The goal is to be able to come back later and remember:
- what embeddings are
- how cosine similarity works
- how to compare words and sentences
- how Sentence Transformers and Gemini embeddings are used
- how text can be split into chunks
- how cosine similarity can retrieve relevant chunks
- how retrieved context can be passed to an LLM

The code is intentionally simple. Run it, change it, and experiment with it.


## 0. Setup

We use:
- `sentence-transformers` for local embeddings
- Google's `google-genai` SDK for Gemini generation and embeddings

**GitHub safety:** never commit a real API key. Enter it at runtime.


In [ ]:
!pip install -q sentence-transformers google-genai


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


## 1. First embedding

A sentence can be converted into a vector — a list of numbers.

`all-MiniLM-L6-v2` produces a 384-dimensional embedding.


In [ ]:
vec1 = model.encode(
    "hello there",
    normalize_embeddings=True
)

print(vec1)
print("Shape:", vec1.shape)


## 2. Tiny vector math

For:

`A = [1, 2]`

`B = [2, 4]`

the dot product is:

`(1 × 2) + (2 × 4) = 10`

The same basic operation is performed across all dimensions of an embedding.


In [ ]:
a = [1, 2]
b = [2, 4]

dot_product = a[0] * b[0] + a[1] * b[1]

print("Dot product =", dot_product)


## 3. Cosine similarity — from scratch

Cosine similarity compares the direction of two vectors.

The function is written manually here so the calculation is visible.


In [ ]:
def cosine_similarity(vec1, vec2):
    numerator = 0
    denom1 = 0
    denom2 = 0

    for cmp1, cmp2 in zip(vec1, vec2):
        numerator += cmp1 * cmp2
        denom1 += cmp1 ** 2
        denom2 += cmp2 ** 2

    return numerator / ((denom1 ** 0.5) * (denom2 ** 0.5))


In [ ]:
vec1 = model.encode(
    ["hello there", "unity and unreal"],
    normalize_embeddings=True
)

cosine_similarity(vec1[0], vec1[1])


### Playground

Change the two sentences and watch the similarity change.

A similarity score is not a universal percentage of meaning. It depends on the embedding model and the texts being compared.


In [ ]:
sentences = [
    "hello there",
    "hi"
]

vectors = model.encode(
    sentences,
    normalize_embeddings=True
)

print(cosine_similarity(vectors[0], vectors[1]))


## 4. Word similarity

Compare every unique pair of words.


In [ ]:
words = ["dog", "cat", "computer", "collar"]

vectors = model.encode(
    words,
    normalize_embeddings=True
)


In [ ]:
for i in range(len(words)):
    for j in range(i + 1, len(words)):
        similarity = cosine_similarity(vectors[i], vectors[j])
        print(words[i], "-", words[j], "=", similarity)


A high similarity score does not mean two things are the same.

For example, `dog` and `collar` can be semantically related because they occur in related contexts.


## 5. Sentence similarity

These sentences are deliberately interesting:
- two are about battery life
- one uses similar words but changes the meaning with `not`
- one is unrelated


In [ ]:
SENTENCES = [
    "My laptop battery drains quickly.",
    "The computer does not last long without charging.",
    "My laptop battery does not drain quickly.",
    "I enjoy playing football on weekends.",
]


In [ ]:
vectors = model.encode(
    SENTENCES,
    normalize_embeddings=True
)

for i in range(len(SENTENCES)):
    for j in range(i + 1, len(SENTENCES)):
        similarity = cosine_similarity(vectors[i], vectors[j])
        print(f"{i} - {j} = {similarity}")
        print("   ", SENTENCES[i])
        print("   ", SENTENCES[j])


## 6. Gemini: generation vs. embeddings

**Generation:** give Gemini text and get generated text back.

**Embedding:** give an embedding model text and get a vector back.

They are different operations.


In [ ]:
from google import genai
import getpass

api_key = getpass.getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=api_key)


In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Who are you?"
)

print(response.text)


In [ ]:
result = client.models.embed_content(
    model="gemini-embedding-2",
    contents="Who are you?"
)

print("Embedding dimensions:", len(result.embeddings[0].values))


## 7. Gemini embeddings + cosine similarity

Same cosine-similarity idea, different source for the vectors.


In [ ]:
def gemini_embeddings(texts):
    result = client.models.embed_content(
        model="gemini-embedding-2",
        contents=texts
    )
    return [embedding.values for embedding in result.embeddings]


In [ ]:
vectors = gemini_embeddings(SENTENCES)

for i in range(len(SENTENCES)):
    for j in range(i + 1, len(SENTENCES)):
        similarity = cosine_similarity(vectors[i], vectors[j])
        print(f"{i} - {j} = {similarity}")


## 8. Corpus → chunks → embeddings

Now move from individual sentences to a document.

The first approach is deliberately simple: **fixed-size word chunks**.

For each chunk we store:

`[embedding, chunk_text]`

The embedding lets us compare it. The original text lets us retrieve and show the actual context later.


In [ ]:
corpus = 'corpus = """\nPortlight Build Day is a one-day community prototyping event at the fictional Tidehouse Learning Hub on 18 October 2026, and its organizing team expects 120 participants working in teams of two to four rather than competing as individuals. Check-in opens at 9:00 a.m. at the East Gate desk and closes at 9:35 a.m.; anyone arriving after that should report to the same desk for a late badge and the missed safety briefing before joining a team. Mira, the operations lead, gives the welcome at 9:45 a.m., the safety briefing begins at 10:05 a.m., and teams choose a project track at 10:20 a.m. before the main building period starts at 10:30 a.m. The first batch of printed leaflets said that final demos would begin in Coral Hall at 4:00 p.m., but Mira sent a correction at 8:10 a.m. on event day after Coral Hall\'s projector failed: the current demo location is Blue Studio on Level 2, and demos begin at 4:15 p.m. The correction did not change the 3:40 p.m. submission deadline, so a team should not treat the later demo start as extra building time. There are three suggested tracks, although teams may combine ideas from them. MapMates is for prototypes that help people plan neighborhood journeys; its starter kit contains a fictional street map, wheelchair-access tags, and a small downloadable route pack so a demo can run without internet access, but the organizers warn that the sample routes have never been certified for real-world navigation. PantryPal is for reducing household food waste using a sample pantry inventory, recipe cards, and expiry dates; its allergen fields are deliberately incomplete, so a prototype may flag uncertainty but must not give definitive medical or dietary advice. StudySpark is for helping learners organize revision from sample class notes in English and Malayalam; the supplied notes contain no real student records, and teams must not upload actual classmates\' marks or personal information. The MapMates route pack has two versions in circulation: version 1 was copied onto a few USB drives before the event, while version 2 was posted to the local files page at 11:40 a.m. after Nila noticed that version 1 placed the East Gate ramp on the wrong side of the building. Teams should use version 2 for their demonstrations and label the old file as superseded if they discuss it. The hub\'s West Door has stairs and closes at 3:00 p.m. for maintenance, while the East Gate has a step-free ramp and a lift to Level 2; the shuttle drops off only at the East Gate every twenty minutes and continues until 5:50 p.m. The exact shuttle driver\'s name and telephone number are not listed in this brief. A quiet room called Lotus is open on Level 1 from 10:30 a.m. until 5:00 p.m.; it is intended for breaks, not presentations, and a participant may bring a support person there. Four volunteer mentors rotate through Help Island near the main worktables from 11:00 a.m. to 12:30 p.m. and again from 2:00 p.m. to 3:15 p.m.: Adi helps with offline data and caching, Nila reviews accessibility and inclusive flows, Rohan helps with search, RAG, and source citations, and Kavya helps teams explain their prototypes clearly. The desk also has twelve loaner laptops, issued first come, first served until 10:00 a.m.; a team can share one, and all loaners and battery packs must return to Help Island by 5:50 p.m. The original lunch sign pointed to the Courtyard from 12:30 p.m. to 1:15 p.m., but a weather update at 11:30 a.m. moved lunch indoors to Palm Café on Level 1 while keeping the same time window. Green labels identify vegan meals, purple labels identify sealed nut-free meals, and participants who need a reserved allergy-aware meal must tell the East Gate desk by 11:00 a.m.; after that cutoff staff can offer fruit but cannot promise a suitable prepared meal. There is a water refill point beside Palm Café, and no food is allowed in Lotus. The event\'s Wi-Fi serves model downloads and the local submission page, but it may become congested when everyone uploads at once. Teams may call a paid external AI API if they have their own key and permission to send the data, but the core prototype must still show a useful path in airplane mode, and judges will disconnect it for thirty seconds during the demo. This offline rule is why the sample datasets and starter code are available locally; it does not mean every feature must work offline or that teams must pay for an API. The organizers will not supply API keys, and teams should keep secrets out of notebooks they share. At 3:40 p.m. a team must submit one ZIP file to the local Portlight submission page containing a README that names the user problem and intended user, the source code or notebook, and one screenshot of the working prototype. A thirty-second video is welcome but optional; a public GitHub repository is not required, and a late email is not a substitute for the ZIP. If the Wi-Fi or submission page fails, teams can hand a USB drive with the same ZIP to Help Island before 3:40 p.m.; the fallback does not extend the deadline. Each team then has three minutes to demo and two minutes for questions in Blue Studio. Judges score usefulness to the intended user at 40 percent, accessibility and inclusion at 25 percent, reliability including the offline path at 20 percent, and clarity at 15 percent; they do not award points for having the most lines of code, the fanciest model, or the earliest submission. A public idea wall outside Blue Studio shows anonymous one-sentence project summaries by default, and a team must opt in before its members\' names or contact details appear. Recognition begins at 5:15 p.m., the event closes at 5:40 p.m., and teams should return borrowed equipment before leaving. The brief does not state a cash prize, a Wi-Fi password, which team will win, how many vegan meals remain after lunch begins, or whether any MapMates route is safe to follow in the real world.\n"""\n'


In [ ]:
embedding_store = list()

chunk = 150

for i in range(0, len(corpus.split()), chunk):
    corpus_chunk = " ".join(corpus.split()[i:i + chunk])

    embedding_store.append([
        model.encode(
            corpus_chunk,
            normalize_embeddings=True
        ),
        corpus_chunk
    ])

print("Number of chunks:", len(embedding_store))


In [ ]:
# Manually inspect a chunk.
chunk_number = 0

print(embedding_store[chunk_number][1])


## 9. Semantic retrieval

Given a query:

1. embed the query
2. compare it with every stored chunk
3. calculate cosine similarity
4. return the chunk with the highest score


In [ ]:
def find_relevant_context(query: str):
    similarity_result = []

    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    for embedding, chunk_text in embedding_store:
        similarity = cosine_similarity(
            embedding,
            query_embedding
        )

        similarity_result.append([
            similarity,
            chunk_text
        ])

    return max(
        similarity_result,
        key=lambda x: x[0]
    )[1]


In [ ]:
query = "what is the portlight build day?"

print(find_relevant_context(query))


### See all retrieval scores

This is useful while learning because you can see *why* a chunk was selected.


In [ ]:
def show_similarity_scores(query: str):
    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    results = []

    for i, (embedding, chunk_text) in enumerate(embedding_store):
        similarity = cosine_similarity(
            embedding,
            query_embedding
        )
        results.append((similarity, i, chunk_text))

    results.sort(reverse=True)

    for similarity, i, chunk_text in results:
        print(f"Chunk {i}: {similarity:.4f}")
        print(chunk_text[:300], "...")
        print()

show_similarity_scores(query)


## 10. Retrieval + LLM = simple RAG

The basic pipeline is:

`query → embedding → retrieve relevant chunk → give context to LLM → answer`

This is intentionally a simple version. No vector database, reranker, or framework is needed to understand the core idea.


In [ ]:
def ask_llm(query: str):
    context = find_relevant_context(query)

    prompt = f'''
For the query given by the user:

Query: {query}

Use the relevant context below to answer the query.
If the context does not contain enough information, say so.

Context:
{context}

Answer:
'''

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text


In [ ]:
print(ask_llm("What time does Portlight Build Day start?"))


## 11. Chunking experiment: sentence boundaries

Fixed word chunks are easy, but they can cut a sentence or related idea in half.

A simple experiment is to keep sentences together and put a small number of sentences into each chunk.

This is **not a full semantic chunker**. It is just a small step beyond fixed word counts using the same ideas already learned.


In [ ]:
corpus = 'corpus = """ On 19 November 2031, the fictional Tideglass Expedition opens at Nila Reef Field Station on the invented island of Velan. The crew\'s public question is simple: why did observers report a blue shimmer in the water on three recent nights? Three explanations are being considered: a plankton bloom, light reflected by pale mineral grains, or a camera-and-sensor artefact. The expedition is not trying to declare a winner from a single bright photograph. It must assemble a small, traceable collection of measurements and samples, then tell visitors what the evidence supports and what remains uncertain. The three study sites are Lantern Bay, Mirror Shelf, and Silt Garden. In this exercise, a “glow index” is an arbitrary number from 0 to 100 generated by a fictional camera-processing pipeline; it is not a standardized scientific unit, and a higher index alone does not identify the cause of light. All times in this brief are station local time.\n\nThe morning briefing begins at 9:00 a.m., field collection starts at 9:30 a.m., and the station laboratory accepts samples until 5:15 p.m. Analysis notes must be locked in the shared report by 6:00 p.m., even if a later public presentation starts late. The original printed schedule placed the public briefing on West Deck at 6:00 p.m. A 4:10 p.m. operations update superseded that line after the deck projector failed: the briefing now begins at 6:30 p.m. in Blue Gallery, which has level access from the station entrance. The update did not move the laboratory cutoff or the 6:00 p.m. report lock. Visitors may watch a demonstration of the evidence board, but they will not handle the sample jars. A crew member who finds a late result after report lock may mention it as an unverified follow-up, not silently edit the locked conclusion. The gallery closes at 7:15 p.m.\n\nLantern Bay is an eight-minute walk east of the station along a marked boardwalk and is the only study site reachable without the skiff. Mirror Shelf is a twenty-minute skiff trip from the station and has an exposed landing. Silt Garden is twelve minutes away by skiff and has a sheltered landing. Those are one-way travel times from the station, not travel times between sites; the brief supplies no direct Mirror-to-Silt route. A skiff trip includes the licensed pilot, so a team cannot assume that every passenger place is available for researchers. The station\'s step-free path to Blue Gallery begins at the main entrance, not at West Deck. A paper map printed before the weather update still shows the original Mirror Shelf afternoon visit. The map is useful for identifying names but not for deciding whether a site is currently open. No exact skiff fuel consumption or fare is provided in this brief.\n\nThe observation kit includes buoy T-4 for water temperature, camera Lumen-2 for the glow index, a handheld salinity meter, and paired clear and opaque sample jars. The opaque jar is the dark control: it travels with the clear jar but is shielded from light. Each field record should name the site, collection time, instrument or jar ID, operator, and whether the value is raw or corrected. A camera frame without a field record can illustrate the scene but cannot by itself establish a sample\'s origin. The salinity meter was checked against a station reference at 8:45 a.m. and remained within the exercise tolerance; no later salinity correction was issued. T-4 and Lumen-2 do have separate correction notices elsewhere in this brief. The team\'s field tablet stores observations locally and syncs when the station network is available, so a missing upload is not the same as a missing local record.\n\nT-4\'s first calibration card, version 1, claimed the buoy read correctly all morning. At 12:10 p.m., sensor lead Nikhil compared it with a reference thermometer and issued calibration card version 2: every T-4 temperature recorded **before 12:10 p.m.** is two degrees Celsius too high, so subtract 2.0 °C from its raw value; readings from 12:10 onward already use the corrected display and must not be adjusted again. The raw Mirror Shelf entry at 11:20 a.m. is 24.6 °C, making its corrected temperature 22.6 °C. The raw Lantern Bay entry at 10:05 a.m. is 23.1 °C, making its corrected temperature 21.1 °C. A Silt Garden reading displayed at 2:05 p.m. is 22.4 °C and remains 22.4 °C. The version 1 card is kept in the archive to show the correction history, not as the current rule.\n\nLumen-2\'s camera clock had a different defect. Photos taken before 11:40 a.m. show a timestamp seven minutes ahead of station time; subtract seven minutes from those displayed photo times when matching them to field records. At 11:40 a.m. the clock was reset, so later photo timestamps are already correct. For example, the Mirror Shelf photograph displayed as 11:32 a.m. was actually taken at 11:25 a.m.; the Lantern Bay photograph displayed as 10:19 a.m. was taken at 10:12 a.m. A photo shown as 2:08 p.m. at Silt Garden was taken at 2:08 p.m. The clock correction changes when a photo was taken, not the glow index calculated from its pixels. The team keeps both displayed and corrected times in its audit table so visitors can see why a photograph and a jar label initially seemed out of order.\n\nFor a jar pair to count as primary sample evidence, the clear and opaque jars must be collected no more than five minutes apart, reach laboratory intake within ninety minutes of the clear jar\'s collection, and have both the collector and intake technician sign the chain-of-custody card at that pair\'s intake handover. If a signature is added after that handover, the sample may be discussed as provisional but cannot be counted as primary evidence. A delayed or unmatched sample is not repaired merely by a convincing photograph. The laboratory can receive a sample until 5:15 p.m.; that closing time does not waive the ninety-minute rule. The expedition labels a usable pair by the clear jar\'s ID, with the control ID recorded beside it. Samples are never tasted or released to visitors. These are the exercise\'s evidence rules, not instructions for real scientific sampling.\n\nThe field log lists three jar pairs. At Mirror Shelf, clear jar M-2 was collected at 11:26 a.m., opaque control MC-2 at 11:28 a.m., and both arrived at intake at 12:45 p.m.; the collector signed at collection, but the intake technician added a missing signature only at 1:05 p.m., after intake. At Silt Garden, clear jar S-1 was collected at 2:40 p.m., control SC-1 at 2:43 p.m., and both arrived at intake at 4:20 p.m. with both signatures already present. At Lantern Bay, clear jar L-3 was collected at 3:25 p.m., control LC-3 at 3:27 p.m., and both reached intake at 4:30 p.m. with both signatures present. The logs also include an unpaired clear jar labelled L-4 at 4:02 p.m.; its opaque control was never collected. The laboratory recorded all four clear jars but did not certify all of them as primary evidence.\n\nThe camera\'s highest logged glow index was 74 at Mirror Shelf at actual station time 11:25 a.m.; the original photo display read 11:32 a.m. Lantern Bay had an index of 31 at 10:12 a.m. and 43 at 3:24 p.m.; Silt Garden had an index of 38 at 2:08 p.m. These numbers came from a processing pipeline that was the same at all three sites, but the source does not provide uncertainty intervals for the index. A separate microscope note reports moving plankton-like cells in the Lantern Bay L-3 clear jar and none in its paired dark control, while the Silt Garden S-1 jar showed bright mineral grains under the same lamp. The Mirror Shelf M-2 jar looked cloudy, but its delayed intake signature limits its evidentiary status. These observations can support comparisons; they do not prove that one mechanism caused every blue shimmer report.\n\nA field ethics rule keeps the crew away from the marked bird-nursery strip at Silt Garden from 3:00 to 5:00 p.m. and forbids shining the high-power Lumen-2 lamp toward nesting areas at any time. The low-power camera mode may be used from the sheltered Silt Garden landing before 3:00 p.m., but a brighter image is not a reason to cross the strip. Lantern Bay\'s boardwalk remains open during the exercise, provided the crew stays on it; Mirror Shelf\'s exposed landing is subject to the weather decision. A visitor-facing visualization may use the supplied map, but it must label the routes as fictional and must not imply that the map is a real navigation or accessibility guarantee. Blue Gallery has level access, yet the source does not say that either skiff landing is wheelchair-accessible. This distinction matters when a public summary describes who can visit each place.\n\nThe 9:00 a.m. plan reserved a second Mirror Shelf trip for 2:00 p.m. At 1:10 p.m., operations lead Asha received a wind warning and canceled all further Mirror Shelf landings for the day; teams already back at the station could still analyze morning Mirror data. The afternoon skiff allocation moved to Silt Garden until 2:50 p.m., allowing the crew to finish before the bird-nursery restriction starts at 3:00 p.m. Lantern Bay remained available by boardwalk until 4:45 p.m. A note copied from the morning plan still says “Mirror, 2:00”; it is an obsolete plan, not permission to go there. The weather notice does not erase valid observations collected before 1:10 p.m., change the sample-evidence rules, or extend the lab and report deadlines. The source does not predict when Mirror Shelf will reopen on a future day.\n\nThe station network was scheduled to be available all day, but a local router failure at 3:35 p.m. stopped tablet sync and the usual web upload. The local tablet records remained readable offline. The contingency is to copy the locked report PDF and its evidence table to the station\'s labeled USB drive and hand it to Diya at the report desk before 6:00 p.m.; Diya records the handover time and later uploads it when the network returns. A team must not put private access tokens or passwords in the PDF, evidence table, or shared notebook. The router failure does not turn a missing field observation into evidence, does not allow a late report, and does not prevent the Blue Gallery briefing from using the offline evidence board. The brief does not state the station Wi-Fi password or exactly when the router will be repaired.\n\nThe expedition roles are deliberately separate. Nikhil owns instrument calibration and can explain the temperature and clock corrections; Asha owns site closures and the field ethics decision; Diya curates the evidence table, verifies citations, and accepts USB report handovers; Kabir is the skiff pilot and alone authorizes a landing after checking operations notices; Reva presents the public briefing and checks whether a claim is understandable without overstating it. Each person can point participants to the relevant record, but no one person\'s memory overrides a signed correction notice or a timed field log. Help is available at the station desk from 10:30 a.m. to noon and 2:30 to 5:00 p.m. The brief gives no personal phone numbers or private email addresses. A visitor should not be told to contact the pilot directly for an unscheduled trip.\n\nThe shared report has three labeled sections: “What we observed,” “What we corrected or excluded,” and “What remains uncertain.” Each factual claim in the first two sections needs a citation to a paragraph, field-log row, or correction card; the public briefing may simplify language but not silently upgrade provisional evidence into primary evidence. The evidence table records a raw value, corrected value when applicable, original timestamp, corrected timestamp when applicable, sample status, and the reason for any exclusion. Diya\'s validation script checks for those columns, not for a particular conclusion about the blue shimmer. The report must be exported as one PDF with the evidence table attached or embedded and locked by 6:00 p.m. A colorful slide deck is optional and cannot replace the PDF. The public board should disclose that the glow index is invented for this exercise and that the map is fictional.\n\nA short summary drafted at 12:00 p.m. said, “Mirror Shelf proves that the shimmer is plankton, because it had the brightest photo and a warm temperature of 24.6 °C.” That sentence is not the expedition\'s approved conclusion. The photo time needs its seven-minute correction, the 11:20 a.m. T-4 temperature needs the two-degree correction, and the M-2 sample is provisional because its intake signature arrived late. The brightest index also has no uncertainty interval or causal meaning on its own. By late afternoon the crew can say that several observations are compatible with different explanations and that the primary Lantern Bay pair contains moving plankton-like cells; it cannot claim that every site has the same cause. The brief does not report a DNA test, a mineral-composition test, or a controlled experiment that separates all three hypotheses.\n\nThe final tabletop challenge asks teams to advise a new crew member who arrives at 3:40 p.m. with only the morning map and the noon summary. By then Mirror Shelf is closed, the station router has failed, and the member has not seen the instrument correction cards. A good assistant must identify which records are superseded, find what can still be done at Lantern Bay, determine which jar pairs qualify as primary evidence, and explain how to submit and present without inventing a definitive scientific answer. It should also distinguish a late but discussable result from a valid primary sample, and an unavailable answer from a negative answer. No monetary prize, precise skiff fuel budget, future reopening time for Mirror Shelf, or identity of the first person to see the shimmer is stated anywhere in this brief. If asked for one of those facts, the assistant should say the source does not establish it."""\n\n'


In [ ]:
import re

sentences = re.split(
    r'(?<=[.!?])\s+',
    corpus.strip()
)

chunk = 5


In [ ]:
embedding_store = list()

for i in range(0, len(sentences), chunk):
    corpus_chunk = " ".join(sentences[i:i + chunk])

    embedding_store.append([
        model.encode(
            corpus_chunk,
            normalize_embeddings=True
        ),
        corpus_chunk
    ])

print("Number of chunks:", len(embedding_store))


In [ ]:
for i, (_, chunk_text) in enumerate(embedding_store):
    print("\n" + "=" * 60)
    print("CHUNK", i)
    print("=" * 60)
    print(chunk_text)


## 12. Playground checklist

### Embeddings
- Change the words.
- Change the sentences.
- Compare related vs. unrelated text.
- Try sentences with similar words but opposite meaning.

### Cosine similarity
- Try your own two vectors.
- Compare all pairs in a list.
- Find the highest-scoring pair.

### Chunking
- Try different word chunk sizes.
- Try different sentence chunk sizes.
- Compare which version retrieves better context.

### Retrieval
- Change the query.
- Run `show_similarity_scores()` and inspect the ranking.
- Try a query whose answer is near a chunk boundary.

### RAG
- Ask something definitely contained in the corpus.
- Ask something not contained in the corpus.
- Notice what happens when the retrieved context is insufficient.


## Quick mental model

**Text**
→ **Embedding**
→ **Vector**
→ **Cosine similarity**
→ **Retrieve relevant text**
→ **Give retrieved text to LLM**
→ **Answer**

For chunking:

**Document**
→ **Chunks**
→ **Embedding for each chunk**
→ **Store `[embedding, text]`**
→ **Compare query embedding with chunk embeddings**
→ **Retrieve context**
